### Install New Libraries

In [80]:
#!pip install ddgs trafilatura -q # -q without any logs

### Load APIs and Libraries

In [81]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from pprint import pprint
import json
from ddgs import DDGS
import trafilatura
from IPython.display import display, Markdown
import contextlib
import io

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API Key is Missing")

client = OpenAI()
MODEL="gpt-4o-mini"
#MODEL="gpt-4.1"

In [82]:
def search_web(query):
    """ Search the web using DuckDuckGo browser. Return 3 results."""
    ddgs = DDGS()
    results = ddgs.text(query, max_results=10)
    print(f"\u2705 got results")
    return json.dumps(results, indent=2)

In [83]:
# Stand alone
url = "https://en.wikipedia.org/wiki/Artificial_intelligence_in_healthcare"
url1 = "https://www.linkedin.com/posts/kazi-jannatun-nayeem_healthcare2030-digitalhealth-artificialintelligence-activity-7481350569320734721-yqoJ"
url2 = "https://www.weforum.org/stories/2025/08/ai-transforming-global-health/"
downloaded = trafilatura.fetch_url(url2)
#print(downloaded)
print(f"\n=====================\n")
if downloaded:
    content = trafilatura.extract(
        downloaded,
        # include_links = True,
        # include_tables=True
    )
    print(content)

In [84]:
def fetch_url(url):
    """Fetch the content of a URL using trafilatura"""
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f" \u2705 Got text: {len(text)} chars")
            return text
    print(f"\u274c Failed to fetch or extract test fron {url}.")
    return f"Could not extract the text from {url}. try a different source"

In [85]:
search_web("Apples in Madagascar")

✅ got results


'[\n  {\n    "title": "Field Guide to Fruit on Madagascar",\n    "href": "https://www.jurgenenkatja.nl/en/fruit-on-madagascar/",\n    "body": "It is also called the Malagasy apple. The outside is hard and you have to work to break the fruit open. This species originates from the Comoros and Madagascar."\n  },\n  {\n    "title": "The Madagascar Apple - Tasting Rare Fruit in New... - YouTube",\n    "href": "https://www.youtube.com/watch?v=v4fyp0gzBVY",\n    "body": "Episode: 786 Madagascar AppleSpecies: Mimusops coriaceaLocation: Noumea, New CaledoniaBig thank you to @fruitaddict for sharing this fruit with me. And also ..."\n  },\n  {\n    "title": "Madagascar :: Custard apple | Custard apple - madagascar, Beautiful...",\n    "href": "https://www.pinterest.com/pin/madagascar-custard-apple--101823641551422176/",\n    "body": "Two custard apples placed on a white plate, accompanied by two spoons. The custard apples have a textured, green exterior with a rough surface."\n  },\n  {\n    "ti

In [86]:
fetch_url("https://www.weforum.org/stories/2025/08/ai-transforming-global-health/")

❌ Failed to fetch or extract test fron https://www.weforum.org/stories/2025/08/ai-transforming-global-health/.


'Could not extract the text from https://www.weforum.org/stories/2025/08/ai-transforming-global-health/. try a different source'

### Step 2: Describe as LLM Tool calling

In [87]:
tools = []

In [88]:
search_web_function = {
    "name": "search_web",
    "description": "Search the web using DuckDuckGo browser. Return 3 results.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to find the relavent websites"
            }
        },
        "required": ["query"]
    }
}

tools.append({"type": "function", "function":search_web_function})

In [89]:
fetch_url_function = {
    "name": "fetch_url",
    "description": "Fetch the extract the main content from a webpage",
    "parameters": {
        "type": "object",
        "properties": {
            "url": {
                "type": "string",
                "description": "The URL of the webpage to fetch and extract the texts"
            }
        },
        "required": ["url"]
    }
}

tools.append({"type": "function", "function":fetch_url_function})

In [90]:
tools

[{'type': 'function',
  'function': {'name': 'search_web',
   'description': 'Search the web using DuckDuckGo browser. Return 3 results.',
   'parameters': {'type': 'object',
    'properties': {'query': {'type': 'string',
      'description': 'The search query to find the relavent websites'}},
    'required': ['query']}}},
 {'type': 'function',
  'function': {'name': 'fetch_url',
   'description': 'Fetch the extract the main content from a webpage',
   'parameters': {'type': 'object',
    'properties': {'url': {'type': 'string',
      'description': 'The URL of the webpage to fetch and extract the texts'}},
    'required': ['url']}}}]

### Step 3: Tool Call Handler

In [91]:
def handle_tool_call(tool_calls):
    tool_results = []

    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        print(f" \U0001f527 Calling funtion: {function_name} with arguments {args}")

        # Route the tool call to the appropropriate function based on the function name
        if function_name == "search_web":
        # Search the Web using the tool
            result = search_web(args["query"])
            content =  f"searched web: {result}"
            print(f" searched web: {result}")
        elif function_name == "fetch_url":
            # Call the second function here
            result = fetch_url(args["url"])
            content = f"URL content: {result}"
            print(f" URL content {result}")
        else:
            content = f"Unknown function: {function_name}"

        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id 
        }
        #print(f"Tool call result: {tool_call_result}")
        tool_results.append(tool_call_result)
    
    # return what to add to our "context" (about tool call results), a dictionary.
    return tool_results

### Step 4: The System Prompt
#### This tells the LLM who it is and how to behave. 
#### The key things:
- what its job is
- What tool it has
- what process to follow
- what output formay to produce

In [92]:
RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

IMPORTANT: The word "DONE:" is a control signal, not a label. Never use it as a heading, section marker, or inline annotation. 
ONLY use the word "DONE:" as per the instructions below -- it has to come at the start of a reply.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 6 different sources, synthesize into a research brief

You MUST gather information from at least 6 distinct sources before delivering your brief. 
If you have fewer than 4 sources, keep searching.

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

### Step 5: The Agentic Loop

In [93]:
def run_reseach_agent(topic: str, max_iterations: int = 10) -> str:
    """
    Run the research agent on a topic and return the reseach brief.
    Args:
        topic: The topic to research
        max_iterations: Safety limit to prevent the infinite loops

    Returns:
        The research brief as a string
    """
    print(f"\n\U0001F50D Starting the research on {topic}")

    # Intialize conversation Message list with system_prompt + Research Task
    messages = [
        {"role": "system", "content": RESEARCH_AGENT_PROMPT},
        {"role": "user", "content": f"Research the following topic and produce a comprehensive research brief:\n {topic}"}
    ]


    # Loop
    iteration = 0

    while iteration < max_iterations:
        iteration +=1
        print(f"\n Iteration: {iteration}")

        #1. Call the LLM and get response
        response = client.chat.completions.create(
            model=MODEL,
            messages = messages,
            tools = tools
        )
        message = response.choices[0].message
        messages.append(message)

        print("*"*15)
        print(message)
        print("*"*15)

        #2 Check if LLM called tools
        if message.tool_calls:
            tool_results = handle_tool_call(message.tool_calls)
            messages.extend(tool_results)
        

        #3 Otherwise: No tools were called, read message content
        else:
            content = message.content
            # Check if Done, then return
            print("="*60)
            print(content)
            if content.startswith("DONE:"):
                research_brief = content[len("DONE:"):].strip()
                print(f"\n\u2705 Research Complete")
                return research_brief
                

            # Otherwise: not yet done, append message
            else:
                print(f" \U0001F4AD Agent is thinking")
                pprint(content)
                # Loop continous to next iteration
        
        #4. If we are entering the final iteration, force a final answer
        if(iteration == max_iterations - 1):
            print("  \u26a0 Safety limited reached. Stopping research in next iterations")
            messages.append({"role": "user", "content":"You have reached the maximum number of iterations. Please deliver your release brief now. You MUST respond with DONE: followed by your brief."})

    # Fallback return
    return "Reseaech incomplete, max iterations reached without finalizing the brief"

### Run

In [94]:
#MODEL="gpt-5.4-mini"
# brief = run_reseach_agent("AI in healthcare in 2030")
# display(Markdown(brief))

In [105]:
TOPICS = [
    "Impact of GLP-1 drugs on the food and beverage industry",
    "State of solid-state EV battery commercialization in 2026",
    "Effects of remote work policies on commercial real estate markets",
    "Geopolitical risks to Taiwan's semiconductor supply chain",
    "Efficacy and adoption of four-day work week pilot programs",
    "Compare RAG and fine-tuning for building an enterprise customer-support assistant.",
    "What are the major causes and consequences of coral reef bleaching?",
    "How did the printing press influence European society during the Renaissance?",
    "What factors have driven electric vehicle adoption in the United States?",
    "What does current research say about the effects of remote work on productivity?"
]
# TOPICS = [
#     "Impact of GLP-1 drugs on the food and beverage industry"
# ]

### Evals

In [96]:
JUDGE_PROMPT_CLAUDE= """
# Judge Prompt: Insufficient Source Breadth (TRUE/FALSE)

You are scoring a research agent's **deliverable research brief** to determine if there was a **source breadth failure**.
Return **only** `TRUE` or `FALSE`.

## Definitions

**Source:** Any distinct, identifiable origin of information referenced or drawn upon in the brief — e.g., a named publication, website, report, dataset, academic paper, company filing, interview, or other external reference. Count by distinct origin, not by number of citations, footnotes, or in-text mentions. Two references to the same outlet/document (e.g., two different articles from the same news site, or two pages of the same report) count as **one** source unless they are clearly independent, separately-published works (e.g., two different articles from the same outlet on different dates/topics can count as two if the brief treats them as separate origins of information).

**Note:** This check is about breadth of research, **not formal citation correctness**. Do not evaluate whether sources are cited in a particular format, whether links are valid, or whether attribution follows a style guide. Simply determine how many distinct sources the brief appears to have drawn information from, based on in-text references, mentions, links, or clear paraphrased attributions (e.g., "according to X," "a report from Y found...", a hyperlink, a named outlet/author).

**Source breadth failure (label TRUE):** Any of the below occurred:

1. **Fewer than 6 distinct sources referenced:** Counting all identifiable distinct sources in the brief (per the definition above), the total is 5 or fewer.
2. **Padded/duplicate sourcing:** The brief appears to reference 6+ sources, but on inspection several are the same underlying source counted multiple times (e.g., same article cited under different phrasing, same outlet's homepage referenced repeatedly) such that the true distinct count is fewer than 6.
3. **Unattributed claims masquerading as researched content:** Large portions of the brief present factual claims, statistics, or findings with no discernible source at all, and removing those unsupported sections would leave fewer than 6 distinct sources underlying the brief's substantive content.
4. **Single-source dependence on a topic requiring breadth:** The brief's core argument or majority of its content relies on 1-2 sources, with only token/passing references to additional sources that don't meaningfully contribute information to the brief.

**No source breadth failure (label FALSE):** The brief draws on **6 or more distinct, independent sources**, each contributing meaningful information (not just a passing or decorative mention), regardless of formatting or citation style.

## Instructions

- Read through the entire brief and enumerate every distinct source you can identify (named outlets, authors, reports, datasets, studies, sites, etc.).
- Deduplicate sources that are the same underlying origin.
- Count only sources that contribute actual information/content to the brief (not sources listed but never actually used).
- If the distinct count is 6 or greater, label FALSE. If 5 or fewer, label TRUE.

## Output Format

Return exactly one token: `TRUE` or `FALSE`. No explanations.
"""

In [97]:
def judge_brief(brief: str) -> str:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": JUDGE_PROMPT_CLAUDE.format(brief=brief)}],
        temperature=0,
        max_tokens=5,
    )
    return resp.choices[0].message.content.strip().upper()

results = []
for topic in TOPICS:
    with contextlib.redirect_stdout(io.StringIO()): # Supress Print output from the agent for clener evaluation logs.
        brief = run_reseach_agent(topic, max_iterations=10)
    verdict = judge_brief(brief)
    results.append({"topic": topic, "verdict": verdict, "brief_len": len(brief)})
    print(f"[{verdict}] {topic}")

failures = sum(1 for r in results if r["verdict"] == "TRUE")
print(f"\n{failures}/{len(results)} topics failed source breadth check")

[FALSE] Impact of GLP-1 drugs on the food and beverage industry

0/1 topics failed source breadth check


In [98]:
JUDGE_PROMPT_OPENAI = """
# Judge Prompt: Insufficient Source Breadth (TRUE/FALSE)

You are scoring a research assistant “transcript/trace” to determine whether the final research brief failed to reference at least 6 different sources.

Return only `TRUE` or `FALSE`.

## Definitions

**Deliverable research brief:** The assistant’s final research response provided to the user. Evaluate only the final deliverable, not sources mentioned exclusively in intermediate search steps, tool calls, planning notes, or hidden reasoning.

**Referenced source:** A source that is explicitly identified in the final research brief through at least one of the following:

- A URL
- A linked citation
- A numbered or footnote citation that maps to an identifiable source
- A clearly named article, webpage, report, paper, publication, organization, or document

Formal citation style is not required.

**Distinct source:** A separately identifiable source, such as a unique webpage, article, report, paper, publication, or document.

Counting rules:

1. Count each distinct source only once, even if it is referenced multiple times.
2. Repeated citations or repeated URLs to the same source count as one source.
3. Different pages, articles, reports, or documents may count as separate sources, even when they come from the same website or organization.
4. A general mention such as “research shows,” “industry reports,” or “according to experts” does not count unless a specific source is identifiable.
5. Do not infer or invent a source that is not explicitly referenced in the final brief.
6. Do not evaluate whether the source supports the claim, is authoritative, or is cited in the correct format. This test measures only the number of distinct sources referenced.

**Insufficient source breadth failure (label TRUE):** The final research brief references fewer than 6 distinct sources.

Examples:

- 0–5 distinct sources referenced → `TRUE`
- Six or more distinct sources referenced → `FALSE`
- Six citations that all point to the same source → `TRUE`
- Six distinct URLs or otherwise identifiable documents → `FALSE`
- Sources appeared in tool calls but fewer than six are referenced in the final brief → `TRUE`

**No source breadth failure (label FALSE):** The final research brief explicitly references at least 6 distinct sources.

## Output Format

Return exactly one token: `TRUE` or `FALSE`. No explanations.
"""

In [106]:

import re
import pandas as pd
results = []

for topic in TOPICS:
    try:
        #with contextlib.redirect_stdout(io.StringIO()):
        brief = run_reseach_agent(topic, max_iterations=10)

        urls = set(re.findall(r"https?://[^\s)>]+", brief))
        source_failure = len(urls) < 6

        results.append({
            "topic": topic,
            "source_count": len(urls),
            "source_failure": source_failure,
            "passed": not source_failure
        })

    except Exception as e:
        results.append({
            "topic": topic,
            "source_count": 0,
            "source_failure": True,
            "passed": False,
            "error": str(e)
        })

df = pd.DataFrame(results)
display(df)

print(f"\nPassed: {df['passed'].sum()}/{len(df)}")


🔍 Starting the research on Impact of GLP-1 drugs on the food and beverage industry

 Iteration: 1
***************
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_pZn4WI4MQHFcg69ejuLMsnRn', function=Function(arguments='{"query":"Impact of GLP-1 drugs on food and beverage industry"}', name='search_web'), type='function')])
***************
 🔧 Calling funtion: search_web with arguments {'query': 'Impact of GLP-1 drugs on food and beverage industry'}
✅ got results
 searched web: [
  {
    "title": "GLP-1's Impact on Food Industry Trends Ahead - GV Wire",
    "href": "https://gvwire.com/2026/02/18/big-food-pours-millions-into-rebrands-as-obesity-drugs-reshape-us-demand/",
    "body": "Major food and beverage companies are reshaping products and portion sizes as the rapid rise of GLP-1 weight-loss drugs changes consumer eating habits. With about 20% of U.S. household

,topic,source_count,source_failure,passed
0,Impact of GLP-1 drugs on the food and beverage...,6,False,True
1,State of solid-state EV battery commercializat...,5,True,False
2,Effects of remote work policies on commercial ...,3,True,False
3,Geopolitical risks to Taiwan's semiconductor s...,4,True,False
4,Efficacy and adoption of four-day work week pi...,5,True,False
5,Compare RAG and fine-tuning for building an en...,6,False,True
6,What are the major causes and consequences of ...,6,False,True
7,How did the printing press influence European ...,5,True,False
8,What factors have driven electric vehicle adop...,3,True,False
9,What does current research say about the effec...,9,False,True



Passed: 4/10
